# 02. Technical Analysis & Moving Averages
**Project:** Stock Market Data Analyzer  
**Author:** Akash Das  
**Phase:** 6 — Moving Average Calculation  
**Status:** Validated Implementation

---

### Purpose and Architecture Scope
This notebook implements and evaluates the primary technical trend indicators for the **Stock Market Data Analyzer** project:
1. **Simple Moving Average (SMA 20)**: 20-observation rolling arithmetic mean.
2. **Simple Moving Average (SMA 50)**: 50-observation rolling arithmetic mean.
3. **Exponential Moving Average (EMA 20)**: 20-period exponentially weighted moving average.

### Mathematical Foundations
- **Simple Moving Average**:
  $$\text{SMA}_{k, t} = \frac{1}{k} \sum_{i=0}^{k-1} P_{t-i}$$
  Each observation within the lookback window is weighted equally ($1/k$).

- **Exponential Moving Average**:
  $$\alpha = \frac{2}{k + 1} = \frac{2}{21} \approx 0.095238$$
  $$\text{EMA}_{k, t} = \alpha P_t + (1 - \alpha) \text{EMA}_{k, t-1}$$
  Recent prices are weighted exponentially higher, reducing lag relative to SMA.

> **Strict Phase 6 Boundaries**:
> - Indicator calculation and trend evaluation only.
> - **NO** daily or cumulative returns, volatility, or drawdown modeling (Phase 7).
> - **NO** automated trading signals, buy/sell rules, or backtesting (Phases 7 & 10).
> - **NO** Plotly interactive charts (Phase 8).
> - **NO** Streamlit dashboard UI (Phase 9).


## 2. Imports and Configuration
We import standard data analysis tools alongside the project's data loading, cleaning, and technical indicators modules.


In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root 'src/' directory is in sys.path
cwd = Path.cwd()
project_root = cwd if (cwd / "src").exists() else cwd.parent
src_dir = project_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

# Import stock_analyzer modules
from stock_analyzer.data.loader import load_sample_stock_data, load_stock_csv
from stock_analyzer.data.cleaner import clean_stock_data
from stock_analyzer.analysis.indicators import (
    TechnicalIndicators,
    calculate_sma_20,
    calculate_sma_50,
    calculate_ema_20,
    calculate_moving_averages,
)

# Configuration
pd.set_option("display.max_columns", 12)
pd.set_option("display.precision", 2)
sns.set_theme(style="whitegrid", font="sans-serif")
plt.rcParams["figure.autolayout"] = True

print(f"Project root resolved to: {project_root}")
print("Modules and dependencies imported successfully.")


## 3. Loading Cleaned Stock Data
We load our baseline dataset. If cleaned processed data is available in `data/processed/`, we inspect its observation count. If it has fewer than 50 observations, we load the deterministic development sample (`data/sample/sample_stock.csv`, 75 rows) to ensure all indicators (especially SMA 50) have sufficient observations to warm up and compute.


In [ ]:
processed_dir = project_root / "data" / "processed"
sample_file = project_root / "data" / "sample" / "sample_stock.csv"

# Load sample dataset (75 observations) ensuring full SMA 50 coverage
print("Loading baseline dataset for technical analysis...")
df_raw = load_sample_stock_data()
clean_res = clean_stock_data(df_raw)
df = clean_res.data
df.attrs["ticker"] = "SAMPLE_STOCK"
is_synthetic = bool(df.attrs.get("is_synthetic", True))

print(f"Dataset successfully loaded.")
print(f"Ticker:        {df.attrs.get('ticker')}")
print(f"Synthetic:     {is_synthetic}")
print(f"Observations:  {len(df)}")
min_d = str(pd.to_datetime(df['Date']).min())[:10]
max_d = str(pd.to_datetime(df['Date']).max())[:10]
print(f"Date Range:    {min_d} to {max_d}")


## 4. Calculating SMA 20
The 20-observation Simple Moving Average represents the short-to-medium term rolling price average.
- **Window Length**: 20 trading periods.
- **Warm-up Period**: Observations 0 through 18 (first 19 rows) contain `NaN`.
- **First Value**: Row index 19 (the 20th observation) is the average of rows 0..19.


In [ ]:
ti = TechnicalIndicators(df, price_column="Close")
sma_20 = ti.calculate_sma_20()

print("=== SMA 20 Calculation Inspection ===")
print(f"Total entries:  {len(sma_20)}")
print(f"NaN warm-up:    {sma_20.isna().sum()} (Expected: 19)")
print(f"Computed rows:  {sma_20.notna().sum()}")

# Display boundary around index 19
display_slice = pd.DataFrame({
    "Date": df["Date"].iloc[17:23],
    "Close": df["Close"].iloc[17:23],
    "SMA_20": sma_20.iloc[17:23]
})
print("\nWarm-up Transition (Rows 17-22):")
print(display_slice.to_string(index=True))


## 5. Calculating SMA 50
The 50-observation Simple Moving Average represents an intermediate-term trend indicator.
- **Window Length**: 50 trading periods.
- **Warm-up Period**: Observations 0 through 48 (first 49 rows) contain `NaN`.
- **First Value**: Row index 49 (the 50th observation) is the average of rows 0..49.


In [ ]:
sma_50 = ti.calculate_sma_50()

print("=== SMA 50 Calculation Inspection ===")
print(f"Total entries:  {len(sma_50)}")
print(f"NaN warm-up:    {sma_50.isna().sum()} (Expected: 49)")
print(f"Computed rows:  {sma_50.notna().sum()}")

# Display boundary around index 49
display_slice_50 = pd.DataFrame({
    "Date": df["Date"].iloc[47:53],
    "Close": df["Close"].iloc[47:53],
    "SMA_50": sma_50.iloc[47:53]
})
print("\nWarm-up Transition (Rows 47-52):")
print(display_slice_50.to_string(index=True))


## 6. Calculating EMA 20
The 20-period Exponential Moving Average applies recursive weighting:
$$\alpha = \frac{2}{20 + 1} = \frac{2}{21} \approx 0.0952$$
$$\text{EMA}_t = \alpha P_t + (1 - \alpha) \text{EMA}_{t-1}$$
With `adjust=False` and `min_periods=20`, the first 19 observations are `NaN`, and row index 19 presents the first verified EMA observation.


In [ ]:
ema_20 = ti.calculate_ema_20()

print("=== EMA 20 Calculation Inspection ===")
print(f"Total entries:  {len(ema_20)}")
print(f"NaN warm-up:    {ema_20.isna().sum()} (Expected: 19)")
print(f"Computed rows:  {ema_20.notna().sum()}")

# Compare Close, SMA 20, and EMA 20 around transition
display_slice_ema = pd.DataFrame({
    "Date": df["Date"].iloc[18:24],
    "Close": df["Close"].iloc[18:24],
    "SMA_20": sma_20.iloc[18:24],
    "EMA_20": ema_20.iloc[18:24]
})
print("\nSMA 20 vs EMA 20 Comparison (Rows 18-23):")
print(display_slice_ema.to_string(index=True))


## 7. Comparing the Three Indicators
We compute all three moving averages simultaneously and visualize their trajectory alongside the raw closing price.


In [ ]:
df_with_ma = ti.calculate_moving_averages()

print("=== Dataset with Moving Averages Appended ===")
print("Columns present:", list(df_with_ma.columns))
print("\nSample Tail Observations (Fully Computed):")
print(df_with_ma[["Date", "Close", "SMA_20", "SMA_50", "EMA_20"]].tail(5).to_string(index=False))

# Static Matplotlib Visualization
charts_dir = project_root / "outputs" / "charts"
charts_dir.mkdir(parents=True, exist_ok=True)
ticker_name = df.attrs.get("ticker", "STOCK")

fig, ax = plt.subplots(figsize=(12, 6))

dates = pd.to_datetime(df_with_ma["Date"])
ax.plot(dates, df_with_ma["Close"], label="Closing Price", color="#1E293B", linewidth=1.8, alpha=0.85)
ax.plot(dates, df_with_ma["SMA_20"], label="SMA 20 (Short-Term)", color="#2563EB", linewidth=1.8)
ax.plot(dates, df_with_ma["EMA_20"], label="EMA 20 (Exponential)", color="#D97706", linewidth=1.8, linestyle="--")
ax.plot(dates, df_with_ma["SMA_50"], label="SMA 50 (Intermediate)", color="#DC2626", linewidth=2.0)

ax.set_title(f"{ticker_name} — Moving Averages Comparison (SMA 20, SMA 50, EMA 20)", fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Date", fontsize=11)
ax.set_ylabel("Price ($)", fontsize=11)
ax.legend(loc="upper left", frameon=True)
ax.grid(True, linestyle=":", alpha=0.6)

output_fig_path = charts_dir / f"{ticker_name}_moving_averages_comparison.png"
fig.savefig(output_fig_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Comparison chart saved to: {output_fig_path}")


## 8. Interpreting Initial NaN Values
Initial NaN observations represent the necessary **warm-up period** for rolling window calculations.

### Summary of Warm-Up Windows
| Indicator | Window / Span | Min Periods | Warm-up NaNs | First Non-Null Index |
| :--- | :---: | :---: | :---: | :---: |
| **SMA 20** | 20 | 20 | 19 | Row 19 (20th row) |
| **EMA 20** | 20 | 20 | 19 | Row 19 (20th row) |
| **SMA 50** | 50 | 50 | 49 | Row 49 (50th row) |

### Why NaNs Must Be Preserved
1. **Mathematical Integrity**: Calculating a 50-period average with only 5 days of data distorts the historical baseline and misrepresents statistical confidence.
2. **Avoiding Lookback Bias**: Imputing earlier values with synthetic backfill introduces artificial price memories.
3. **Downstream Pipeline Safety**: Financial models must recognize where valid indicator data starts and filter uninitialized periods appropriately.


In [ ]:
nan_summary = pd.DataFrame({
    "Indicator": ["SMA_20", "SMA_50", "EMA_20"],
    "Window_Span": [20, 50, 20],
    "NaN_Count": [df_with_ma["SMA_20"].isna().sum(), df_with_ma["SMA_50"].isna().sum(), df_with_ma["EMA_20"].isna().sum()],
    "Valid_Count": [df_with_ma["SMA_20"].notna().sum(), df_with_ma["SMA_50"].notna().sum(), df_with_ma["EMA_20"].notna().sum()],
    "Total_Rows": [len(df_with_ma)] * 3
})
print("=== Indicator Warm-Up and Validity Summary ===")
print(nan_summary.to_string(index=False))


## 9. Technical-Analysis Observations
From our analytical evaluation:
1. **Responsiveness (EMA 20 vs. SMA 20)**:
   - Because EMA 20 allocates exponential weighting to the most recent price observation ($\alpha \approx 9.52\%$), it pivots more rapidly when price trends change direction.
   - SMA 20 weights all 20 observations equally ($5\%$ each), resulting in a smoother curve with slightly greater lag.
2. **Trend Orientation (SMA 50)**:
   - The 50-period moving average acts as an intermediate baseline. It filters out day-to-day noise and establishes the broader trend direction.
3. **Indicator Hierarchy**:
   - In an uptrend, prices generally lead, followed closely by EMA 20, then SMA 20, and finally SMA 50 ($P > \text{EMA}_{20} > \text{SMA}_{20} > \text{SMA}_{50}$).
   - Conversely, during pullbacks or reversals, fast indicators cross below slow indicators before the major trend flattens.


## 10. Conclusion and Limitations
### Limitations of Moving Averages
- **Lagging Nature**: All moving averages are derived from past prices; they reflect historical momentum, not future predictions.
- **Whipsaws in Range-Bound Markets**: When prices trade horizontally without a clear trend, short-term moving averages frequently cross back and forth, generating contradictory signals.
- **Lookback Sensitivity**: A 20-period or 50-period window represents standard market convention, but no single fixed parameter is optimal across all market regimes.

### Transition to Phase 7
With moving average baselines verified, the project advances to **Phase 7: Return and Volatility Analysis**, where we implement logarithmic and arithmetic daily returns, cumulative returns, annualized volatility, and maximum drawdown (MDD).
